In [22]:
#########################################
# Imports
#########################################

from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

from langchain_core.output_parsers import StrOutputParser
from langchain.tools import tool

In [23]:

# LLM


llm = Ollama(model="mistral")


In [25]:
#########################################
# Knowledge Base
#########################################

with open("medassist_knowledge_base.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

documents=[text_data]

In [26]:
#########################################
# Chunking
#########################################

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = text_splitter.create_documents(documents)



In [27]:

#########################################
# Embeddings
#########################################

embeddings = OllamaEmbeddings(model="mistral")


In [28]:
#########################################
# Vector Store
#########################################

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever()

In [29]:
#########################################
# Retriever Runnable
#########################################

def retrieve_docs(query):

    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    return context


retrieval_runnable = RunnableLambda(retrieve_docs)

In [ ]:



# #########################################
# # TOOLS
# #########################################

# @tool
# def weather_tool(city: str):
#     """Return weather information for a city."""

#     mock_weather = {
#         "Paris": "18°C Cloudy",
#         "Tokyo": "22°C Sunny",
#         "Rome": "25°C Clear"
#     }

#     return mock_weather.get(city, "Weather data not available")


# @tool
# def travel_cost_tool(destination: str):
#     """Return estimated travel cost from India."""

#     mock_costs = {
#         "Italy": "₹80k - ₹1.2L",
#         "Japan": "₹90k - ₹1.5L",
#         "Thailand": "₹30k - ₹60k"
#     }

#     return mock_costs.get(destination, "Cost data not available")


In [30]:
#########################################
# Prompt Builder
#########################################

def build_prompt(inputs):

    context = inputs["context"]
    query = inputs["query"]

    prompt = f"""
You are a helpful medical assistant.
Answer qiestion using the hospital knowledge base.

Use the context to answer the question.

Context:
{context}

Question:
{query}
"""

    return prompt


prompt_runnable = RunnableLambda(build_prompt)

In [31]:
#########################################
# RAG Pipeline
#########################################

rag_chain = (
    {
        "context": retrieval_runnable,
        "query": RunnablePassthrough()
    }
    | prompt_runnable
    | llm
    | StrOutputParser()
)

In [32]:
#########################################
# SESSION MEMORY
#########################################

store = {}

def get_session_history(session_id):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]


In [33]:
#########################################
# Wrap chain with memory
#########################################

chat_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="query",
    history_messages_key="chat_history"
)

In [34]:
#########################################
# Entity Tracking
#########################################

entity_store = {}

def update_entities(session_id, query):

    if session_id not in entity_store:
        entity_store[session_id] = {
            "disease": None,
            "diet": None,
            "allergy": None,
        }

    q = query.lower()

    if "diabetes" in q:
        entity_store[session_id]["diet"] = "vegetarian"

    if "vegan" in q:
        entity_store[session_id]["diet"] = "non vegetarian"

    if "italy" in q:
        entity_store[session_id]["disease"] = "diabetes"

    if "japan" in q:
        entity_store[session_id]["disease"] = "hypertension"

    if "thailand" in q:
        entity_store[session_id]["disease"] = "heart disease"


In [35]:
#########################################
# Ask Assistant
#########################################

def ask_assistant(session_id, query):

    update_entities(session_id, query)

    q = query.lower()

    response = chat_chain.invoke(
        {"query": query},
        config={"configurable": {"session_id": session_id}}
    )

    return response



In [36]:
print(ask_assistant("user1", "What are symptoms of diabetes?"))

 Diabetes, specifically Type 1 and Type 2, can present with various symptoms. Some common symptoms include:

1. Increased thirst and frequent urination: This happens because the body is trying to get rid of the excess glucose (sugar) in the blood by urinating more frequently.

2. Increased hunger: Due to the body's inability to use glucose effectively, it requires more food for energy, leading to increased appetite.

3. Fatigue: Fatigue or weakness can occur due to the body's inability to provide adequate energy to the cells.

4. Blurred vision: High blood sugar levels can affect the lens of the eye, causing blurred vision.

5. Slow healing of cuts or wounds: Diabetes can impair the body's ability to fight infections and heal, leading to slow wound healing.

6. Numbness or tingling in the hands or feet: This is known as neuropathy and can occur due to damage to the nerves from high blood sugar levels.

7. Frequent infections: Diabetes can lower the body's resistance to infections, lead